In [1]:
from google.cloud import bigquery

# The client will now automatically find your credentials via ADC
client = bigquery.Client(project="methodical-mark-493108-d4")
print("Authenticated via ADC")

Authenticated via ADC


In [2]:
import pandas as pd
def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query on BigQuery and return a pandas DataFrame."""
    job = client.query(sql)
    return job.to_dataframe()

In [3]:
df_sepsis = run_query("""
SELECT * FROM `physionet-data.mimiciv_3_1_derived.sepsis3` LIMIT 1000;
""")
df_sepsis.head()

c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_id,stay_id,antibiotic_time,culture_time,suspected_infection_time,sofa_time,sofa_score,respiration,coagulation,liver,cardiovascular,cns,renal,sepsis3
0,18212223,30752269,2136-08-15 08:00:00,2136-08-14 23:00:00,2136-08-14 23:00:00,2136-08-15 04:00:00,2,0,0,0,0,0,2,True
1,13736311,39454408,2178-04-29 08:00:00,2178-04-28 15:45:00,2178-04-28 15:45:00,2178-04-29 09:00:00,2,0,0,0,0,1,1,True
2,19085966,34095671,2156-02-03 04:00:00,2156-02-03 03:52:00,2156-02-03 03:52:00,2156-02-03 17:00:00,2,1,0,0,0,1,0,True
3,10147182,33175266,2179-11-23 00:00:00,2179-11-22 21:08:00,2179-11-22 21:08:00,2179-11-22 21:00:00,2,2,0,0,0,0,0,True
4,19669410,38510130,2143-10-02 20:00:00,2143-10-03 00:19:00,2143-10-02 20:00:00,2143-10-02 19:00:00,2,0,0,0,0,2,0,True


In [ ]:
import torch

In [3]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA Capability: (8, 9)


In [16]:
import json
import pandas as pd
import numpy as np

df = pd.read_csv('full_data.csv') 

mira_data = []
# GROUP BY BOTH STAY AND ITEM
for (stay_id, itemid), group in df.groupby(['stay_id', 'itemid']):
    group = group.sort_values('hrs_since_anchor')
    
    # MIRA needs to know WHAT this variable is. 
    # We use 'feat_id' so the Graph-RAG can identify it later.
    vals = group['valuenum'].fillna(0.0).tolist()
    times = group['hrs_since_anchor'].tolist()
    mask = [1] * len(vals)

    patient_var_entry = {
        "stay_id": str(stay_id),
        "itemid": str(itemid), # CRITICAL for Graph-RAG lookup
        "label": 1 if str(group['label'].iloc[0]).lower() == 'true' else 0,
        "sequence": vals,
        "time": times,
        "mask": mask
    }
    mira_data.append(patient_var_entry)

with open('train_mira_kaggle.jsonl', 'w') as f:
    for entry in mira_data:
        f.write(json.dumps(entry) + '\n')

In [2]:
import torch
import json
from mira.mira.models.modeling_mira import MIRAForPrediction

# 1. Load your local model
model_path = "mira/checkpoints"
model = MIRAForPrediction.from_pretrained(model_path).cuda()
model.eval()

c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0424 10:30:20.537000 27872 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


MIRAForPrediction(
  (model): MIRAModel(
    (embed_layer): MIRAInputEmbedding(
      (emb_layer): Linear(in_features=1, out_features=384, bias=False)
      (gate_layer): Linear(in_features=1, out_features=384, bias=False)
      (act_fn): SiLU()
    )
    (layers): ModuleList(
      (0-11): 12 x MIRADecoderLayer(
        (self_attn): MIRAAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
          (rotary_emb): ContinuousTimeRotaryEmbedding()
        )
        (ffn_layer): MIRASparseExpertsLayer(
          (gate): Linear(in_features=384, out_features=8, bias=False)
          (experts): ModuleList(
            (0-7): 8 x MIRATemporalBlock(
              (gate_proj): Linear(in_features=384, out_features=768, bias=False)
              (up_p

In [3]:
import torch
import json
import numpy as np
from mira.mira.models.modeling_mira import MIRAForPrediction
from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP & MAPPINGS
# Using the itemid dictionary you provided earlier
ITEM_MAP = {
    "221289": "Epinephrine", "221662": "Dopamine", "221749": "Phenylephrine",
    "221906": "Norepinephrine", "222315": "Vasopressin", "220210": "Resp Rate",
    "220277": "SpO2", "220045": "Heart Rate", "220052": "MAP", "223762": "Temp (C)"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "mira/checkpoints" # Point this to your saved folder
model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

def run_forecast(itemid, history_vals, history_times, steps_to_forecast=3):
    """
    Takes a history of one variable and predicts the future trajectory.
    """
    label = ITEM_MAP.get(itemid, f"Unknown ({itemid})")
    
    # Convert to Tensors [Batch=1, Length]
    seq = torch.tensor([history_vals], dtype=torch.float32).to(device)
    time = torch.tensor([history_times], dtype=torch.float32).to(device)
    attn_mask = torch.ones_like(seq).to(device)

    # 2. TIME NORMALIZATION (Required for MIRA's CT-RoPE)
    # MIRA needs relative time geometry to understand the gaps
    full_scaled_times, _, _ = normalize_time_for_ctrope(
        time_values=time,
        attention_mask=attn_mask,
        seq_length=seq.shape[1],
        alpha=1.0
    )

    # 3. AUTOREGRESSIVE FORECASTING
    current_vals = seq.clone()
    current_times = full_scaled_times.clone()
    predictions = []

    print(f"\n--- Forecaster Input: {label} ---")
    print(f"Recent History: {history_vals[-3:]} at times {history_times[-3:]}")

    with torch.no_grad():
        for i in range(steps_to_forecast):
            # MIRA expects [Batch, Length, 1] for the values
            inp_vals = current_vals.unsqueeze(-1)
            
            output = model(
                input_ids=inp_vals,
                time_values=current_times,
                return_dict=True
            )
            
            # Get the very last logit (the future prediction)
            next_val = output.logits[:, -1, :]
            predictions.append(next_val.item())

            # Update tensors for the next step in the loop
            current_vals = torch.cat([current_vals, next_val], dim=1)
            
            # Simple time-step increment for the forecast (e.g., +1 hour)
            next_time = current_times[:, -1:] + 1.0 
            current_times = torch.cat([current_times, next_time], dim=1)

    print(f"Predicted Trend: {predictions}")
    return predictions



In [14]:
l = {}
with open('train_mira.jsonl', 'r') as f:
    for line in f:
        entry = json.loads(line)
        itemid = entry['itemid']
        history_vals = entry['sequence']
        history_times = entry['time']
        if itemid not in l:
            l[itemid] = (history_vals, history_times)


for itemid, (history_vals, history_times) in l.items():
    run_forecast(itemid, history_vals, history_times, steps_to_forecast=3)


--- Forecaster Input: Unknown (50885) ---
Recent History: [0.6] at times [0.3333333333333333]
Predicted Trend: [0.5363202691078186, 0.4284543991088867, 0.3084966540336609]

--- Forecaster Input: Unknown (50912) ---
Recent History: [1.0, 0.9] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [0.761544942855835, 0.613304078578949, 0.4709845185279846]

--- Forecaster Input: Unknown (50931) ---
Recent History: [132.0, 119.0] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [14.087804794311523, 5.6179704666137695, 1.2159326076507568]

--- Forecaster Input: Unknown (51265) ---
Recent History: [260.0, 251.0] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [1.7128721475601196, 1.8138915300369263, 2.122591018676758]

--- Forecaster Input: Unknown (51301) ---
Recent History: [21.9, 22.5] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [15.647808074951172, 5.811712741851807, 2.297614574432373]

--- Forecaster Input: Heart R

In [11]:
run_forecast("220045", [72], [0], steps_to_forecast=3)


--- Forecaster Input: Heart Rate ---
Recent History: [72] at times [0]
Predicted Trend: [15.300379753112793, 3.735386848449707, 1.365187644958496]


[15.300379753112793, 3.735386848449707, 1.365187644958496]

In [12]:
# import torch
# import json
# import numpy as np
# import pandas as pd
# from mira.mira.models.modeling_mira import MIRAForPrediction
# from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "mira/checkpoints" 
model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

# Mapping for the output
ITEM_MAP = {
    "221289": "Epinephrine", "221662": "Dopamine", "221749": "Phenylephrine",
    "221906": "Norepinephrine", "222315": "Vasopressin", "220210": "Resp Rate",
    "220277": "SpO2", "220045": "Heart Rate", "220052": "MAP", "223762": "Temp (C)"
}

def run_distinct_forecasts(data_path, steps=3):
    seen_items = set()
    
    with open(data_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            itemid = str(data.get('itemid'))
            
            # Only process each itemid once for the demo
            if itemid in seen_items or itemid not in ITEM_MAP:
                continue
            seen_items.add(itemid)
            
            # Prepare Data
            seq = torch.tensor([data['sequence']], dtype=torch.float32).to(device)
            times = torch.tensor([data['time']], dtype=torch.float32).to(device)
            
            # Stats for Normalization (Crucial for MIRA)
            mean = seq.mean()
            std = seq.std() + 1e-6
            seq_norm = (seq - mean) / std

            # Time Normalization
            full_scaled_times, _, _ = normalize_time_for_ctrope(
                time_values=times,
                attention_mask=torch.ones_like(times),
                seq_length=seq.shape[1],
                alpha=1.0
            )

            # Autoregressive Forecast
            cur_vals = seq_norm.clone()
            cur_times = full_scaled_times.clone()
            preds_norm = []

            with torch.no_grad():
                for _ in range(steps):
                    out = model(input_ids=cur_vals.unsqueeze(-1), time_values=cur_times)
                    next_val_norm = out.logits[:, -1, :]
                    preds_norm.append(next_val_norm.item())
                    
                    # Update for next step
                    cur_vals = torch.cat([cur_vals, next_val_norm], dim=1)
                    next_t = cur_times[:, -1:] + 1.0 # Predict 1 hour ahead
                    cur_times = torch.cat([cur_times, next_t], dim=1)

            # De-normalize
            preds_real = [round((p * std.item()) + mean.item(), 2) for p in preds_norm]
            history = [round(x, 2) for x in data['sequence'][-3:]]

            print(f"[{ITEM_MAP[itemid]}] ID: {itemid}")
            print(f"  > History (last 3): {history}")
            print(f"  > MIRA Forecast:    {preds_real}")
            print("-" * 40)

# 2. EXECUTE
print(f"Running forecasts on unique items from training data...\n")
run_distinct_forecasts("./train_mira.jsonl")

Running forecasts on unique items from training data...

[Heart Rate] ID: 220045
  > History (last 3): [76.0, 77.0, 83.0]
  > MIRA Forecast:    [81.27, 82.33, 84.72]
----------------------------------------
[Resp Rate] ID: 220210
  > History (last 3): [20.0, 17.0, 31.0]
  > MIRA Forecast:    [21.35, 19.03, 18.52]
----------------------------------------
[SpO2] ID: 220277
  > History (last 3): [97.0, 99.0, 97.0]
  > MIRA Forecast:    [98.22, 98.39, 98.44]
----------------------------------------
[MAP] ID: 220052
  > History (last 3): [85.0, 79.0, 81.0]
  > MIRA Forecast:    [82.94, 85.62, 87.67]
----------------------------------------
[Phenylephrine] ID: 221749
  > History (last 3): [2.0, 1.0, 0.5]
  > MIRA Forecast:    [1.82, 1.65, 1.46]
----------------------------------------
[Norepinephrine] ID: 221906
  > History (last 3): [0.03, 0.05, 0.07]
  > MIRA Forecast:    [0.07, 0.06, 0.05]
----------------------------------------
[Temp (C)] ID: 223762
  > History (last 3): [37.3, 37.2, 37

C:\Users\nisha\AppData\Local\Temp\ipykernel_27872\253878398.py:40: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = seq.std() + 1e-6


[Dopamine] ID: 221662
  > History (last 3): [5.0, 6.01, 5.0]
  > MIRA Forecast:    [5.24, 5.32, 5.36]
----------------------------------------
